# Dataset Creation — 5-Fold Cross Validation
> Converts AnnData (.h5ad) into per-donor `.npy` files in a **single flat folder**.
> The CV splits are handled entirely in the training script — no pre-split needed here.

### Pipeline
```
AnnData → donor-level label map → all_labels.npy
                               → data/all/<donor>_embeddings.npy (one per donor)
```

**Key difference from discovery/validation split**
- No `train_test_split` at this stage
- All donors go into `data/all/` with a single `data/all_labels.npy`
- `StratifiedKFold` in the training script handles the 5-fold CV split at donor level

In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc

print(f"scanpy version : {sc.__version__}")

## 1 — Load AnnData

In [ ]:
# ── Change to your file path ──────────────────────────────────
adata = sc.read_h5ad("SCP1884_subset_50_normalised_expressions_with_embeddings_cp.h5ad")

print(f"Total cells    : {adata.shape[0]}")
print(f"Total features : {adata.shape[1]}")
print(f"obsm keys      : {list(adata.obsm.keys())}")
print(f"obs columns    : {adata.obs.columns.tolist()}")

## 2 — Inspect donors and labels

In [ ]:
print("Unique donors :", adata.obs["donor_id"].nunique())
print("Disease labels:", adata.obs["disease__ontology_label"].unique().tolist())

donor_disease = (
    adata.obs
    .groupby("donor_id")["disease__ontology_label"]
    .first()
    .reset_index()
)
print("\nDonor disease breakdown:")
print(donor_disease["disease__ontology_label"].value_counts())
print()
print(donor_disease.head(10))

## 3 — Configuration

In [ ]:
# ── Edit these values to match your dataset ───────────────────
LABEL_MAP = {
    "normal"          : 0,
    "Crohn's disease" : 1,
}

EMBEDDING_KEY = "X_scGPT"   # check adata.obsm.keys()
DONOR_COL     = "donor_id"
DISEASE_COL   = "disease__ontology_label"
OUTPUT_DIR    = "data/all/"   # ALL donors go here — no pre-split
LABELS_PATH   = "data/all_labels.npy"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output dir  : {OUTPUT_DIR}")
print(f"Labels path : {LABELS_PATH}")
print(f"Embedding   : {EMBEDDING_KEY}")

## 4 — Validate label mapping before extraction

In [ ]:
donor_info = (
    adata.obs
    .groupby(DONOR_COL)[DISEASE_COL]
    .first()
    .reset_index()
)
donor_info.columns = ["donor_id", "disease"]
donor_info["label"] = donor_info["disease"].map(LABEL_MAP)

unmapped = donor_info[donor_info["label"].isna()]
if len(unmapped) > 0:
    print(f"WARNING — unmapped labels: {unmapped['disease'].unique().tolist()}")
    print("Add these to LABEL_MAP before continuing!")
else:
    print("All labels mapped successfully ✓")

print("\nDonor label distribution:")
print(donor_info["label"].value_counts())
print(f"\nTotal donors: {len(donor_info)}")

## 5 — Extraction function

In [ ]:
def extract_all_patients_for_cv(
    adata,
    output_dir,
    labels_path,
    embedding_key = "X_scGPT",
    donor_col     = "donor_id",
    disease_col   = "disease__ontology_label",
    label_map     = None,
):
    """
    Extracts per-donor embeddings into a single flat folder.
    All donors are saved here; CV splitting is done at training time.

    Output structure
    ----------------
    output_dir/
        donor001_embeddings.npy    (num_cells, d_h)
        donor002_embeddings.npy
        ...
    labels_path → (num_donors,) int64 array, sorted by filename

    Returns
    -------
    labels_arr  : np.ndarray int64
    cell_counts : list of int
    """
    os.makedirs(output_dir, exist_ok=True)

    donor_ids   = sorted(adata.obs[donor_col].unique())
    labels      = []
    skipped     = []
    cell_counts = []
    last_Z      = None

    print(f"Extracting {len(donor_ids)} donors → {output_dir}")
    print("-" * 65)

    for did in donor_ids:
        donor_mask  = adata.obs[donor_col] == did
        donor_adata = adata[donor_mask]

        # ── Get disease label ──────────────────────────────────
        raw_label = donor_adata.obs[disease_col].mode()[0]

        if label_map is not None:
            if raw_label not in label_map:
                print(f"  SKIP {did} — '{raw_label}' not in label_map")
                skipped.append(did)
                continue
            numeric_label = label_map[raw_label]
        else:
            try:
                numeric_label = int(raw_label)
            except ValueError:
                print(f"  SKIP {did} — cannot convert '{raw_label}' to int")
                skipped.append(did)
                continue

        # ── Get embedding matrix ───────────────────────────────
        if embedding_key in donor_adata.obsm:
            Z = donor_adata.obsm[embedding_key]          # (num_cells, d_h)
        else:
            print(f"  WARNING: '{embedding_key}' not in obsm — falling back to X")
            Z = (donor_adata.X.toarray()
                 if hasattr(donor_adata.X, "toarray")
                 else np.array(donor_adata.X))

        # ── Quality checks ─────────────────────────────────────
        assert not np.isnan(Z).any(), f"NaN in donor {did}"
        assert not np.isinf(Z).any(), f"Inf in donor {did}"

        # ── Save ───────────────────────────────────────────────
        safe_did = str(did).replace("/", "_").replace(" ", "_")
        fname    = f"{safe_did}_embeddings.npy"
        np.save(os.path.join(output_dir, fname), Z.astype(np.float32))

        labels.append(numeric_label)
        cell_counts.append(Z.shape[0])
        last_Z = Z

        print(f"  {str(did):<35} "
              f"label={numeric_label} ({raw_label:<18}) "
              f"cells={Z.shape[0]:>5}")

    # ── Save labels array ──────────────────────────────────────
    labels_arr = np.array(labels, dtype=np.int64)
    np.save(labels_path, labels_arr)

    # ── Summary ────────────────────────────────────────────────
    print("-" * 65)
    print(f"Saved       : {len(labels_arr)} donors")
    print(f"Skipped     : {len(skipped)} {skipped if skipped else '' }")
    print(f"Labels path : {labels_path}")
    if cell_counts:
        print(f"Cell range  : {min(cell_counts)} – {max(cell_counts)} per donor")
        print(f"Embedding   : d_h = {last_Z.shape[1]}")
    print("\nClass balance:")
    for name, cls in sorted(label_map.items(), key=lambda x: x[1]):
        n = (labels_arr == cls).sum()
        print(f"  {name:<20} (label={cls}) : "
              f"{n:>3} donors ({n/len(labels_arr)*100:.1f}%)")

    return labels_arr, cell_counts

## 6 — Run extraction

In [ ]:
all_labels, all_counts = extract_all_patients_for_cv(
    adata         = adata,
    output_dir    = OUTPUT_DIR,
    labels_path   = LABELS_PATH,
    embedding_key = EMBEDDING_KEY,
    donor_col     = DONOR_COL,
    disease_col   = DISEASE_COL,
    label_map     = LABEL_MAP,
)

## 7 — Verify output integrity

In [ ]:
all_files = sorted([f for f in os.listdir(OUTPUT_DIR) if f.endswith(".npy")])
labels_loaded = np.load(LABELS_PATH)

print("INTEGRITY CHECKS")
print("-" * 50)
print(f"Donor files  : {len(all_files)}")
print(f"Labels array : {len(labels_loaded)}")
assert len(all_files) == len(labels_loaded), "File/label count mismatch!"
print("File ↔ label count match : ✓")

# Check no duplicate donor IDs
donor_name_set = set(f.replace("_embeddings.npy", "") for f in all_files)
assert len(donor_name_set) == len(all_files), "Duplicate donor filenames detected!"
print("No duplicate donor IDs   : ✓")

# Sample shape check
sample_file  = all_files[0]
sample_array = np.load(os.path.join(OUTPUT_DIR, sample_file))
print(f"\nSample file  : {sample_file}")
print(f"Shape        : {sample_array.shape}   (num_cells, d_h)")
print(f"dtype        : {sample_array.dtype}")
print(f"NaN present  : {np.isnan(sample_array).any()}")
print(f"Inf present  : {np.isinf(sample_array).any()}")

## 8 — Preview 5-fold CV splits (dry run, no training)

In [ ]:
from sklearn.model_selection import StratifiedKFold

N_FOLDS     = 5
RANDOM_SEED = 42

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
donor_names = [f.replace("_embeddings.npy", "") for f in all_files]

print(f"5-Fold CV split preview ({N_FOLDS} folds, stratified by label)")
print("=" * 65)

for fold, (train_idx, val_idx) in enumerate(
    skf.split(np.arange(len(all_files)), labels_loaded), start=1
):
    train_labels = labels_loaded[train_idx]
    val_labels   = labels_loaded[val_idx]

    print(f"Fold {fold}:")
    print(f"  Train : {len(train_idx):>3} donors  "
          f"Normal={( train_labels==0).sum():>2}  Crohn={(train_labels==1).sum():>2}")
    print(f"  Val   : {len(val_idx):>3} donors  "
          f"Normal={(val_labels==0).sum():>2}  Crohn={(val_labels==1).sum():>2}")

    # Confirm no donor overlap
    overlap = set(train_idx) & set(val_idx)
    assert len(overlap) == 0, f"Leakage in fold {fold}!"
    print(f"  Overlap: 0 ✓")
    print()

print("All splits verified — no donor leakage ✓")

## 9 — Summary table

In [ ]:
rows = []
for fname, lbl, cnt in zip(all_files, labels_loaded.tolist(), all_counts):
    rows.append({
        "donor_id"   : fname.replace("_embeddings.npy", ""),
        "label"      : lbl,
        "disease"    : {v: k for k, v in LABEL_MAP.items()}[lbl],
        "num_cells"  : cnt,
    })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))
print(f"\nTotal donors : {len(summary_df)}")
print(summary_df["disease"].value_counts().to_string())

summary_df.to_csv("data/all_donors_summary.csv", index=False)
print("\nSaved → data/all_donors_summary.csv ✓")